# E1: 1D CNN for DNA Thermodynamics (Standardized)

**Thesis:** Inductive Biases in Representation Learning for DNA Thermodynamic Property Prediction  
**Experiment ID:** E1  
**Thesis Chapter:** Chapter 2 — Baselines  

## Core Idea
A 1D CNN processes the sequence as a series of overlapping windows, capturing **local translation-invariant** patterns (nearest-neighbour stacking interactions). The 7-channel input encodes both nucleotide identity and secondary structure at each position.

**Input:** `(7, 24)` — 4 one-hot nucleotide channels + 3 one-hot structure channels, channels-first for Conv1d.  
**Inductive bias:** Translation invariance along sequence axis — the same kernel fires wherever a given dinucleotide stacking motif appears.

In [1]:
# ── 1. Setup & Imports ────────────────────────────────────────────────────────
import os, json, time
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

import wandb
import sys
if sys.platform == 'win32' and not os.environ.get('WANDB_MODE'):
    os.environ['WANDB_MODE'] = 'online'

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

COLORS = {'E0_GNN':'#7f8c8d','E1_1DCNN':'#3498db','E2_2DCNN':'#e74c3c',
          'E3_SAT':'#9b59b6','E4_PINN':'#e67e22','E5_Hybrid':'#1abc9c'}
MODEL_COLOR = COLORS['E1_1DCNN']

c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda:0
GPU: NVIDIA GeForce GTX 1660 Ti


In [2]:
# ── 2. Configuration ──────────────────────────────────────────────────────────
MAX_LEN = 24

DATA_CSV   = 'data/models/raw/combined_dataset.csv'
SPLIT_JSON = 'data/models/raw/combined_data_split.json'

config = dict(
    model_name     = '1D_CNN',
    experiment_id  = 'E1',
    in_channels    = 7,
    dropout        = 0.2,
    n_epoch        = 200,
    batch_size     = 256,
    lr             = 1e-3,
    weight_decay   = 1e-5,
    grad_clip      = 1.0,
    dataset        = 'arr',
    wandb_project  = 'NNN_Thesis_Experiments',
    checkpoint_dir = 'MyExperiments/1DCNN/models',
)
print('Config:', config)

Config: {'model_name': '1D_CNN', 'experiment_id': 'E1', 'in_channels': 7, 'dropout': 0.2, 'n_epoch': 200, 'batch_size': 256, 'lr': 0.001, 'weight_decay': 1e-05, 'grad_clip': 1.0, 'dataset': 'arr', 'wandb_project': 'NNN_Thesis_Experiments', 'checkpoint_dir': 'MyExperiments/1DCNN/models'}


In [3]:
# ── 3. Data Loading & Normalization ───────────────────────────────────────────
df = pd.read_csv(DATA_CSV, index_col='SEQID')
df.sort_index(inplace=True)
with open(SPLIT_JSON) as f:
    split = json.load(f)

# Train & evaluate on 'arr' only — lit_uv / ov are held-out generalization sets
TRAIN_DATASET = 'arr'

train_df = df.loc[split['train_ind']].dropna(subset=['dH','Tm'])
train_df = train_df[train_df['dataset'] == TRAIN_DATASET]
val_df   = df.loc[split['val_ind']  ].dropna(subset=['dH','Tm'])
val_df   = val_df[val_df['dataset'] == TRAIN_DATASET]
test_df  = df.loc[split['test_ind'] ].dropna(subset=['dH','Tm'])
test_df  = test_df[test_df['dataset'] == TRAIN_DATASET]

sumstats = {
    'dH_min': float(train_df['dH'].min()), 'dH_max': float(train_df['dH'].max()),
    'Tm_min': float(train_df['Tm'].min()), 'Tm_max': float(train_df['Tm'].max()),
}
def normalize(v, mn, mx):   return (v - mn) / (mx - mn)
def unnormalize(v, mn, mx): return v * (mx - mn) + mn

print(f'Train {len(train_df):,}  Val {len(val_df):,}  Test {len(test_df):,}  (arr only)')
print(f'dH [{sumstats["dH_min"]:.1f}, {sumstats["dH_max"]:.1f}]  Tm [{sumstats["Tm_min"]:.1f}, {sumstats["Tm_max"]:.1f}]')

Train 25,025  Val 1,318  Test 1,387  (arr only)
dH [-68.2, -2.7]  Tm [13.6, 68.6]


In [ ]:
# ── 4. Encoding: 1D sequence + structure → (7, L) channels-first ──────────────

def encode_row_1d(row, max_len=MAX_LEN):
    """Returns (7, max_len) float32 array, channels-first for Conv1d."""
    seq_map  = {'A':0,'T':1,'C':2,'G':3}
    str_map  = {'(':0,')':1,'.':2}
    refseq   = str(row['RefSeq'])
    if '[' in refseq:
        try:    refseq = ''.join(eval(refseq))
        except: pass
    struct = str(row['TargetStruct']).replace('+','')
    L = min(len(refseq), max_len)
    x = np.zeros((max_len, 7), dtype=np.float32)
    for i in range(L):
        if refseq[i].upper() in f: x[i, seq_map[refseq[i].upper()]] = 1.
        if i < len(struct) and struct[i] in str_map: x[i, 4+str_map[struct[i]]] = 1.
    return x.T   # (7, max_len) — channels first

_r = df.iloc[0]
print(f'Encoding shape: {encode_row_1d(_r).shape}  (7 channels × {MAX_LEN} positions)')

Encoding shape: (7, 24)  (7 channels × 24 positions)


In [5]:
# ── 5. Dataset & DataLoaders ──────────────────────────────────────────────────

class DNA1DDataset(Dataset):
    def __init__(self, df, sumstats):
        self.df = df.reset_index(drop=False)
        self.ss = sumstats
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = encode_row_1d(row)
        y = np.array([normalize(row['dH'], self.ss['dH_min'], self.ss['dH_max']),
                      normalize(row['Tm'], self.ss['Tm_min'], self.ss['Tm_max'])], dtype=np.float32)
        return torch.tensor(x), torch.tensor(y)

train_ds = DNA1DDataset(train_df, sumstats)
val_ds   = DNA1DDataset(val_df,   sumstats)
test_ds  = DNA1DDataset(test_df,  sumstats)
train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=512,                  shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=512,                  shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')
_x, _y = next(iter(train_loader))
print(f'Batch — x: {_x.shape}  y: {_y.shape}')

Train batches: 98  Val batches: 3
Batch — x: torch.Size([256, 7, 24])  y: torch.Size([256, 2])


In [6]:
# ── 6. Model: 1D CNN ──────────────────────────────────────────────────────────

class DNA_CNN(nn.Module):
    """
    4-block 1D CNN for DNA thermodynamic regression.
    Conv kernels of size 3 and 5 capture dinucleotide and trinucleotide
    nearest-neighbour stacking contexts.
    """
    def __init__(self, in_channels=7, dropout=0.2):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv1d(in_channels, 64,  kernel_size=3, padding=1), nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(64,          128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(128,         128, kernel_size=5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(128,         128, kernel_size=5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.head(self.pool(self.conv_block(x)).squeeze(-1))


model = DNA_CNN(in_channels=config['in_channels'], dropout=config['dropout']).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: DNA_CNN  |  Parameters: {n_params:,}')

Model: DNA_CNN  |  Parameters: 199,490


In [7]:
# ── 7. Metrics & Evaluation Helpers ──────────────────────────────────────────

def compute_metrics(pred_norm, true_norm, sumstats):
    if torch.is_tensor(pred_norm): pred_norm = pred_norm.cpu().numpy()
    if torch.is_tensor(true_norm): true_norm = true_norm.cpu().numpy()
    dH_p = pred_norm[:,0]*(sumstats['dH_max']-sumstats['dH_min'])+sumstats['dH_min']
    Tm_p = pred_norm[:,1]*(sumstats['Tm_max']-sumstats['Tm_min'])+sumstats['Tm_min']
    dH_t = true_norm[:,0]*(sumstats['dH_max']-sumstats['dH_min'])+sumstats['dH_min']
    Tm_t = true_norm[:,1]*(sumstats['Tm_max']-sumstats['Tm_min'])+sumstats['Tm_min']
    dG_p = dH_p*(1.-(273.15+37.)/(273.15+Tm_p))
    dG_t = dH_t*(1.-(273.15+37.)/(273.15+Tm_t))
    metrics = {}
    for tag, p, t in [('dH',dH_p,dH_t),('Tm',Tm_p,Tm_t),('dG_37',dG_p,dG_t)]:
        mask = np.isfinite(t) & np.isfinite(p)
        if mask.sum() < 2:
            metrics[f'{tag}_mae'] = metrics[f'{tag}_rmse'] = metrics[f'{tag}_r2'] = float('nan')
        else:
            d = p[mask]-t[mask]
            metrics[f'{tag}_mae']  = float(np.mean(np.abs(d)))
            metrics[f'{tag}_rmse'] = float(np.sqrt(np.mean(d**2)))
            metrics[f'{tag}_r2']   = float(r2_score(t[mask], p[mask]))
    return metrics, dH_p, Tm_p, dH_t, Tm_t

@torch.no_grad()
def evaluate(model, loader, sumstats, device):
    model.eval()
    preds, trues = [], []
    for x, y in loader:
        preds.append(model(x.to(device)).cpu()); trues.append(y)
    return compute_metrics(torch.cat(preds), torch.cat(trues), sumstats)

print('Metrics helpers defined.')

Metrics helpers defined.


In [8]:
# ── 8. Training Loop ──────────────────────────────────────────────────────────

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['n_epoch'], eta_min=1e-5)

history = {'train_loss':[], 'val_dH_mae':[], 'val_Tm_mae':[], 'val_dG_mae':[], 'val_dH_rmse':[], 'val_Tm_rmse':[]}
os.makedirs(config['checkpoint_dir'], exist_ok=True)
os.makedirs('out', exist_ok=True)

_run_name = 'E1_1DCNN_4block_k3k5'
_kw = dict(project=config['wandb_project'], name=_run_name, config=config, reinit=True)
_mode = os.environ.get('WANDB_MODE','').strip().lower()
if _mode in ('offline','disabled'):
    run = wandb.init(mode=_mode, **_kw)
else:
    try:    run = wandb.init(**_kw)
    except Exception as e:
        print(f'WandB online failed ({e}), offline.'); run = wandb.init(mode='offline', **_kw)
print(f'WandB run: {run.name}  |  mode: {run.settings.mode}')

best_val_dG = float('inf'); start = time.time()

for epoch in range(config['n_epoch']):
    model.train(); train_loss = 0.
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_loader.dataset)
    scheduler.step()

    vm, *_ = evaluate(model, val_loader, sumstats, device)
    history['train_loss'].append(train_loss)
    history['val_dH_mae'].append(vm['dH_mae'])
    history['val_Tm_mae'].append(vm['Tm_mae'])
    history['val_dG_mae'].append(vm['dG_37_mae'])
    history['val_dH_rmse'].append(vm['dH_rmse'])
    history['val_Tm_rmse'].append(vm['Tm_rmse'])

    wandb.log({'epoch':epoch,'train_loss':train_loss,**{f'val_{k}':v for k,v in vm.items()},'lr':scheduler.get_last_lr()[0]})

    if vm['dG_37_mae'] < best_val_dG:
        best_val_dG = vm['dG_37_mae']
        torch.save(model.state_dict(), os.path.join(config['checkpoint_dir'],'best_cnn1d_model.pt'))

    if (epoch+1) % 20 == 0:
        print(f"Ep {epoch+1:3d}/{config['n_epoch']} | loss {train_loss:.4f} | dH {vm['dH_mae']:.3f} | Tm {vm['Tm_mae']:.3f} | dG {vm['dG_37_mae']:.3f} | {(time.time()-start)/60:.1f}min")

run.finish()
with open('out/cnn1d_history.json','w') as f: json.dump(history, f)
print(f'Done. Best val dG MAE: {best_val_dG:.4f}')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\anant\.netrc.


wandb: Currently logged in as: apati087 (apati087-university-of-california-riverside) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


WandB run: E1_1DCNN_4block_k3k5  |  mode: online
Ep  20/200 | loss 0.0078 | dH 3.577 | Tm 2.712 | dG 0.249 | 1.9min
Ep  40/200 | loss 0.0061 | dH 3.554 | Tm 2.432 | dG 0.247 | 3.6min
Ep  60/200 | loss 0.0054 | dH 3.172 | Tm 2.214 | dG 0.210 | 5.4min
Ep  80/200 | loss 0.0051 | dH 3.169 | Tm 2.215 | dG 0.211 | 7.3min
Ep 100/200 | loss 0.0048 | dH 3.116 | Tm 2.129 | dG 0.201 | 9.1min
Ep 120/200 | loss 0.0044 | dH 3.024 | Tm 1.951 | dG 0.191 | 10.8min
Ep 140/200 | loss 0.0042 | dH 3.014 | Tm 1.878 | dG 0.186 | 12.6min
Ep 160/200 | loss 0.0039 | dH 3.033 | Tm 1.889 | dG 0.190 | 14.3min
Ep 180/200 | loss 0.0038 | dH 2.937 | Tm 1.811 | dG 0.178 | 16.1min
Ep 200/200 | loss 0.0038 | dH 2.932 | Tm 1.832 | dG 0.180 | 17.8min


epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▇▇▇▇▇▇▇▇██
lr,███████▇▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▄▄▄▄▄▄▄▃▂▂▂▂▁▁▁▁▁
train_loss,█▅▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_Tm_mae,█▆▇▄▃▃▃▄▂▃▃▂▂▂▂▂▂▂▃▂▂▂▂▂▂▁▁▂▂▁▁▂▁▁▁▁▁▁▁▁
val_Tm_r2,▁▃▄▅▅▆▆▆▅▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇███████████████
val_Tm_rmse,▇█▆▄▆▃▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁
val_dG_37_mae,█▅▄▄▄▄▃▃▂▂▂▂▃▂▂▂▂▂▂▂▂▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_dG_37_r2,▁▁▃▄▄▅▆▅▅▄▆▇▇▆▇▇▆▇▇▇▆▇▇▇▇▇▇██▇▇█████████
val_dG_37_rmse,██▅▄▄▃▃▂▂▂▃▂▂▂▂▂▂▂▂▂▁▁▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁
val_dH_mae,█▇▅▄▄▄▃▃▃▃▃▃▄▃▃▂▂▃▃▂▂▃▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁
+2,...


Done. Best val dG MAE: 0.1766


In [9]:
# ── 9. Final Evaluation ───────────────────────────────────────────────────────

model.load_state_dict(torch.load(os.path.join(config['checkpoint_dir'],'best_cnn1d_model.pt'), map_location=device))
val_m,  dH_vp, Tm_vp, dH_vt, Tm_vt = evaluate(model, val_loader,  sumstats, device)
test_m, dH_tp, Tm_tp, dH_tt, Tm_tt = evaluate(model, test_loader, sumstats, device)

print('=== Val (arr) ===');  [print(f'  {t}  MAE {val_m[f"{k}_mae"]:.3f}  R2 {val_m[f"{k}_r2"]:.3f}') for t,k in [('dH','dH'),('Tm','Tm'),('dG37','dG_37')]]
print('=== Test (arr) ==='); [print(f'  {t}  MAE {test_m[f"{k}_mae"]:.3f}  R2 {test_m[f"{k}_r2"]:.3f}') for t,k in [('dH','dH'),('Tm','Tm'),('dG37','dG_37')]]

ev = pd.DataFrame({'dH_pred':dH_vp,'dH_true':dH_vt,'Tm_pred':Tm_vp,'Tm_true':Tm_vt})
ev['dG_pred'] = ev['dH_pred']*(1-310.15/(273.15+ev['Tm_pred']))
ev['dG_true'] = ev['dH_true']*(1-310.15/(273.15+ev['Tm_true']))
ev.to_csv('out/cnn1d_val_eval.csv', index=False)

run_log = dict(experiment_id='E1', model='DNA_CNN', config=config,
               n_params=sum(p.numel() for p in model.parameters() if p.requires_grad),
               val_metrics=val_m, test_metrics=test_m,
               best_checkpoint=os.path.join(config['checkpoint_dir'],'best_cnn1d_model.pt'))
with open('out/cnn1d_run_log.json','w') as f: json.dump(run_log, f, indent=2)
print('Saved: out/cnn1d_val_eval.csv  out/cnn1d_run_log.json')

=== Val (arr) ===
  dH  MAE 2.911  R2 0.867
  Tm  MAE 1.808  R2 0.947
  dG37  MAE 0.177  R2 0.938
=== Test (arr) ===
  dH  MAE 2.857  R2 0.877
  Tm  MAE 1.848  R2 0.940
  dG37  MAE 0.178  R2 0.940
Saved: out/cnn1d_val_eval.csv  out/cnn1d_run_log.json


In [10]:
# ── 10. Convergence Curves (F2 contribution) ──────────────────────────────────
os.makedirs('out/figures', exist_ok=True)
ep = range(1, len(history['train_loss'])+1)
fig, axes = plt.subplots(1, 3, figsize=(14,4), facecolor='#f8f9fa')
for ax, (k, yl) in zip(axes, [('val_dH_mae','Val dH MAE (kcal/mol)'),('val_Tm_mae','Val Tm MAE (deg C)'),('val_dG_mae','Val dG37 MAE (kcal/mol)')]):
    ax.plot(ep, history[k], color=MODEL_COLOR, lw=2, label='E1: 1D CNN')
    ax.set_xlabel('Epoch'); ax.set_ylabel(yl); ax.legend(fontsize=9); sns.despine(ax=ax)
fig.suptitle('E1: 1D CNN Convergence', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('out/figures/cnn1d_convergence.png', dpi=300, bbox_inches='tight')
plt.show(); print('Saved: out/figures/cnn1d_convergence.png')

Saved: out/figures/cnn1d_convergence.png


C:\Users\anant\AppData\Local\Temp\ipykernel_40924\3735365496.py:11: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show(); print('Saved: out/figures/cnn1d_convergence.png')


In [11]:
# ── 11. Scatter Plots (F3 contribution) ───────────────────────────────────────
AXIS_LIMITS = {'dH':(-55,-5),'Tm':(20,60),'dG_37':(-7,5)}
dG_vp = dH_vp*(1-310.15/(273.15+Tm_vp)); dG_vt = dH_vt*(1-310.15/(273.15+Tm_vt))
fig, axes = plt.subplots(1, 3, figsize=(14,5), facecolor='#f8f9fa')
for ax, (p,t,tag,unit) in zip(axes, [(dH_vp,dH_vt,'dH','kcal/mol'),(Tm_vp,Tm_vt,'Tm','deg C'),(dG_vp,dG_vt,'dG_37','kcal/mol')]):
    lim = AXIS_LIMITS[tag]; m = np.isfinite(p)&np.isfinite(t)
    ax.scatter(t[m],p[m],s=4,alpha=0.4,color=MODEL_COLOR,rasterized=True)
    ax.plot(lim,lim,'k--',alpha=0.3,lw=1.5)
    mae=np.mean(np.abs(p[m]-t[m])); r2=r2_score(t[m],p[m])
    ax.text(0.05,0.93,f'MAE={mae:.3f}\nR2={r2:.3f}',transform=ax.transAxes,fontsize=8.5,va='top',bbox=dict(boxstyle='round,pad=0.3',fc='white',alpha=0.8))
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_xlabel(f'Measured {tag}'); ax.set_ylabel(f'Predicted {tag}'); ax.set_title(tag,fontweight='bold'); sns.despine(ax=ax)
fig.suptitle('E1: 1D CNN Predicted vs Measured (Val)', fontsize=11)
plt.tight_layout()
plt.savefig('out/figures/cnn1d_scatter.png', dpi=300, bbox_inches='tight')
plt.show(); print('Saved: out/figures/cnn1d_scatter.png')

Saved: out/figures/cnn1d_scatter.png


C:\Users\anant\AppData\Local\Temp\ipykernel_40924\2500626492.py:15: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show(); print('Saved: out/figures/cnn1d_scatter.png')


In [12]:
# ── 12. lit_uv Generalization (F4 contribution) ───────────────────────────────
lit_df = df[df['dataset']=='lit_uv'].copy()
print(f'lit_uv: {len(lit_df)} sequences (Tm-only)')

model.eval(); Tm_preds = []
with torch.no_grad():
    for _, row in lit_df.iterrows():
        x = torch.tensor(encode_row_1d(row)).unsqueeze(0).to(device)
        out = model(x)
        Tm_preds.append(unnormalize(out[0,1].item(), sumstats['Tm_min'], sumstats['Tm_max']))

Tm_true = lit_df['Tm'].values
lit_mae = float(np.mean(np.abs(np.array(Tm_preds) - Tm_true)))
print(f'lit_uv Tm MAE: {lit_mae:.3f} degC')

# Save to run_log
with open('out/cnn1d_run_log.json') as f: rl = json.load(f)
rl['lit_uv_Tm_mae'] = lit_mae
with open('out/cnn1d_run_log.json','w') as f: json.dump(rl, f, indent=2)
print('Updated cnn1d_run_log.json with lit_uv result.')

lit_uv: 348 sequences (Tm-only)
lit_uv Tm MAE: 6.109 degC
Updated cnn1d_run_log.json with lit_uv result.
